# Домашнее задание 7

In [ ]:
# Импорты
import pandas as pd
import numpy as np


from sklearn.ensemble import RandomForestRegressor
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, roc_auc_score


# это может помочь вам для поиска оптимальных gamma в задании №1
from scipy.optimize import minimize

# Для BaseLine в задании №2
from xgboost.sklearn import XGBClassifier


## Градиентный бустинг своими руками (3 балла)

Реализуйте алгоритм градиентного бустинга для регрессии. 

Напомним **основные формулы**.

Обозначим текущую композицию на $K-1$ шаге за $a_{K-1}(x_i)$. Следующий базовый алгоритм $b_K(x_i)$ обучается на ответах $ -\left. \frac{\partial L(y_i, z)}{\partial z} \right|_{z = a_{K-1}(x_i)}$, где $L(y_i, z)$ - значение лосса на объектае при правильном ответе $y_i$ и предсказании $z$.

Композиция на следующем шаге получается следующим образом:

$$a_K(x_i) = a_{K-1}(x_i) + \gamma \eta_K b_K(x_i)$$

Здесь $\gamma$ - гиперпараметр `learning rate`, а $\eta_K$ = оптимальный вес, настраиваемый на каждом шаге алгоритма, который можно найти по следующей формуле (обратите внимание на отсутствие $\gamma$):

$$\eta_k = \arg\min_\eta \sum_{i=1}^N \mathcal{L}(a_{k-1}(x_i) + \eta b_k(x_i), y_i)$$

Можете принять $\eta_K = 1$ для каждого шага, либо реализуйте нахождение оптимального $\gamma_K$ на каждом шаге, чтобы получить ещё **1 балл** за задачу.

В качестве лосса возьмите $L = MSE$

**Примечание**. Вы можете использовать `DecisionTree` из `sklearn` и методы оптимизации из различных библиотек.


За что начисляются баллы:

* 1 балл за верную реализацию класса
* 1 балл за нахождение оптимального коэф. $\gamma$
* 1 балл за подбор гиперпараметров при помощи `GridSearch` или `Optuna`

In [ ]:
class GradientBoosting:
    def __init__(self, n_estimators, max_depth, learning_rate=0.1):
        """
        PARAMETERS:
        n_estimators - number of trees in the ensemble
        max_depth - maximum depth of a tree
        learning_rate - coefficient by which new algorithm result is multiplied
        """
        # your code here

    def fit(self, x, y):
        """
        INPUT:
        x - np.array of shape (k, d)
        y - np.array of shape (k,)
        """
        # Здесь нам нужно проитерироваться по n_estimators и обучить
        # соответствующее количество деревьев с помощью _fit_predict_tree(),
        # правильно обновляя y_new
        # Деревья нужно где-то сохранить, чтобы затем использовать в predict()
        # your code here
        for ...:
            y_new = ...
            # your code here

    def _fit_predict_tree(self, x, y):
        # Обучаем дерево и возвращаем его предикшн
        tree = ...
        # your code here
        return self.gamma * self.learning_rate * tree.predict(x)

    def predict(self, x):
        """
        INPUT:
        x - np.array of shape (m, d)
        OUTPUT:
        y_pred - np.array of shape (m,)
        """
        # Используем сохранённые деревья для расчёта агрегированного предикшна
        # your code here
        return y_pred

Проверьте вашу реализацию на `California Housing`.

Подберите оптимальные гиперпараметры, чтобы победить `RandomForestRegressor` как в обычном случае, так и при нахождении оптимального шага (не меняйте параметры сида). При необходимости воспользуйтесь `GridSearch` или `Optuna`. За это вы получите ещё **1 балл**.

In [2]:
housing  = fetch_california_housing()
X = housing.data
y = housing.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=13)

In [ ]:
# RandomForestRegressor

rf = RandomForestRegressor(max_features=4, n_estimators=640, random_state=19052019)

rf.fit(X_train, y_train)
mean_squared_error(y_test, rf.predict(X_test))
mean_squared_error(y_test, y_pred)

0.2494859271050316

In [ ]:
# Проверьте свою реализацию GradientBoosting
# GradientBoosting

gb = ...

In [ ]:
# Для удобства проверяющего укажите, пожалуйста, реализовали ли вы нахождение оптимального шага?
print("ДА")
print("НЕТ")

## Прогнозируем задержки самолётов (2 балла)

![alt](../hw/airplane.jpg)

```Где найти самые дешёвые билеты на самолет.. знает каждый```

Поработаем с задачей про задержки самолётов. На основании доступных данных о рейсе вам нужно определить, будет ли он задержан на 15 минут. Воспользуйтесь любыми методами градиентного бустинга {XGboost, catboost, LightGBM} и GridSearchCV для достижения результата.

Получите 1 балл за преодоление порога roc_auc_score 0.72 и ещё 1 балл за преодоление порога 0.74.

За что начисляются баллы:
* 1 балл за преодоление порога roc_auc_score 0.72
* ещё 1 балл за преодоление порога 0.74

In [26]:
train = pd.read_csv('flight_delays_train.csv')
test = pd.read_csv('flight_delays_test.csv')

In [27]:
train.head()

,Month,DayofMonth,DayOfWeek,DepTime,UniqueCarrier,Origin,Dest,Distance,dep_delayed_15min
0,c-8,c-21,c-7,1934,AA,ATL,DFW,732,N
1,c-4,c-20,c-3,1548,US,PIT,MCO,834,N
2,c-9,c-2,c-5,1422,XE,RDU,CLE,416,N
3,c-11,c-25,c-6,1015,OO,DEN,MEM,872,N
4,c-10,c-7,c-6,1828,WN,MDW,OMA,423,Y


In [29]:
# Baseline
from xgboost.sklearn import XGBClassifier

X_train = train[['Distance', 'DepTime']].values
y_train = train['dep_delayed_15min'].map({'Y': 1, 'N': 0}).values
X_test = test[['Distance', 'DepTime']].values
X_train_part, X_valid, y_train_part, y_valid = train_test_split(X_train, y_train, test_size=0.3)

xgb_model = XGBClassifier()
xgb_model.fit(X_train_part, y_train_part)
roc_auc_score(y_valid, xgb_model.predict_proba(X_valid)[:, 1])

0.6998626038106989

In [ ]:
# your code here